# 📊 Notebook 1: Exploratory Data Analysis (EDA)

**Môn học:** Học Máy (CO3001) — Học kỳ I (2025-2026)  
**Nhóm:** 13

### 👥 Thành viên
* Hoàng Xuân Bách · Nguyễn Việt Hùng · Đặng Mậu Anh Quân · Cao Lê Minh Khoa · Phạm Hồ Minh Khoa

### Mục tiêu
- Hiểu cấu trúc và đặc điểm của Adult Census Income dataset
- Phát hiện missing values và outliers
- Phân tích phân phối và tương quan giữa các features


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request, warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

for d in ['../reports/figures', 'reports/figures', '../data', 'data']:
    os.makedirs(d, exist_ok=True)

FIG_DIR = '../reports/figures' if os.path.isdir('../reports') else 'reports/figures'
print('✅ Libraries loaded!')


## 2. Load Adult Census Income Dataset

In [ ]:
COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

TRAIN_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
TEST_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

DATA_DIR   = '../data' if os.path.isdir('../data') else 'data'
os.makedirs(DATA_DIR, exist_ok=True)
TRAIN_PATH = os.path.join(DATA_DIR, 'adult_train.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'adult_test.csv')

if not os.path.exists(TRAIN_PATH):
    print('Đang tải dữ liệu từ UCI...')
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)
    urllib.request.urlretrieve(TEST_URL,  TEST_PATH)
    print('✅ Tải xong!')
else:
    print(f'✅ Cache: {TRAIN_PATH}')

df_train = pd.read_csv(TRAIN_PATH, names=COLUMNS, sep=r',\s*', engine='python', na_values='?')
df_test  = pd.read_csv(TEST_PATH,  names=COLUMNS, sep=r',\s*', engine='python', na_values='?', skiprows=1)
df_full  = pd.concat([df_train, df_test], ignore_index=True)
df_full['income'] = df_full['income'].str.replace('.', '', regex=False).str.strip()

print(f'Dataset: {df_full.shape[0]:,} mẫu × {df_full.shape[1]} cột')
df_full.head()


## 3. Thông Tin Cơ Bản

In [ ]:
print('=== Dataset Info ===')
df_full.info()
print('\n=== Basic Statistics (Numeric) ===')
df_full.describe().round(2)


In [ ]:
print(f'Tổng mẫu    : {len(df_full):,}')
print(f'Số features : {df_full.shape[1] - 1}')
print(f'Duplicates  : {df_full.duplicated().sum():,}')
print('\nKiểu dữ liệu:')
print(df_full.dtypes.value_counts())


## 4. Phân Tích Missing Values

In [ ]:
missing = df_full.isnull().sum()
missing_pct = 100 * missing / len(df_full)
missing_df = pd.DataFrame({'Count': missing, 'Percentage (%)': missing_pct.round(2)})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

if missing_df.empty:
    print('✅ Không có missing values!')
else:
    print(f'⚠️  {len(missing_df)} cột có missing values:')
    print(missing_df)
    
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(missing_df.index, missing_df['Percentage (%)'],
                   color=['#e74c3c','#e67e22','#f1c40f'], edgecolor='white')
    for bar, val in zip(bars, missing_df['Percentage (%)']):
        ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}%', va='center', fontsize=11)
    ax.set_xlabel('Missing (%)', fontsize=12)
    ax.set_title('Missing Values by Feature', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()


## 5. Phân Tích Biến Mục Tiêu (Target)

In [ ]:
target_counts = df_full['income'].value_counts()
target_pct    = df_full['income'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2ecc71', '#e74c3c']

bars = axes[0].bar(target_counts.index, target_counts.values,
                   color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Phân phối Target (Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Số mẫu')
for bar, v in zip(bars, target_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+200, f'{v:,}',
                 ha='center', fontweight='bold', fontsize=12)

axes[1].pie(target_counts.values, labels=target_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 13})
axes[1].set_title('Phân phối Target (%)', fontsize=13, fontweight='bold')

plt.suptitle('Adult Census Income — Target Distribution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(target_counts.to_string())
print(f'\n→ Imbalanced: {target_pct.idxmax()} chiếm ~{target_pct.max():.1f}%')


## 6. Phân Tích Numeric Features

In [ ]:
numeric_cols = ['age','fnlwgt','education_num','capital_gain','capital_loss','hours_per_week']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for idx, col in enumerate(numeric_cols):
    for cls, color in zip(['<=50K', '>50K'], ['#3498db', '#e74c3c']):
        axes[idx].hist(df_full[df_full['income']==cls][col].dropna(),
                       bins=35, alpha=0.6, label=cls, color=color, edgecolor='none')
    axes[idx].set_title(col, fontsize=12, fontweight='bold')
    axes[idx].legend(fontsize=9)
    axes[idx].set_ylabel('Count')

plt.suptitle('Phân phối Numeric Features theo Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/numeric_by_target.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Box plots — phát hiện outliers
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for idx, col in enumerate(numeric_cols):
    df_full.boxplot(column=col, by='income', ax=axes[idx],
                    patch_artist=True, boxprops=dict(facecolor='lightblue', alpha=0.7))
    axes[idx].set_title(col, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('')
plt.suptitle('Box Plots — Numeric Features by Income', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Phân Tích Categorical Features

In [ ]:
cat_cols = ['workclass','education','marital_status','occupation',
            'relationship','race','sex','native_country']

fig, axes = plt.subplots(4, 2, figsize=(18, 22))
axes = axes.flatten()
for idx, col in enumerate(cat_cols):
    top_cats = df_full[col].value_counts().head(8).index
    subset   = df_full[df_full[col].isin(top_cats)]
    ct       = pd.crosstab(subset[col], subset['income'], normalize='index') * 100
    ct.plot(kind='barh', ax=axes[idx], color=['#3498db','#e74c3c'],
            edgecolor='white', alpha=0.85)
    axes[idx].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('%')
    axes[idx].legend(title='Income', fontsize=9)

plt.suptitle('Categorical Features — % Income by Category', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/categorical_by_target.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Correlation Heatmap

In [ ]:
df_corr = df_full[numeric_cols].copy()
df_corr['income_binary'] = (df_full['income'] == '>50K').astype(int)
corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, mask=mask,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix (Numeric Features + Target)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# High correlation pairs
threshold = 0.7
high_corr = [(corr.columns[i], corr.columns[j], corr.iloc[i,j])
             for i in range(len(corr.columns))
             for j in range(i+1, len(corr.columns))
             if abs(corr.iloc[i,j]) > threshold]
if high_corr:
    print(f'Các cặp có |corr| > {threshold}:')
    for f1, f2, c in high_corr:
        print(f'  {f1} ↔ {f2}: {c:.3f}')
else:
    print('✅ Không có cặp feature tương quan quá cao.')


## 9. Tổng Kết EDA

In [ ]:
print('=' * 55)
print('TỔNG KẾT EDA — Adult Census Income')
print('=' * 55)
print(f'Tổng mẫu              : {len(df_full):,}')
print(f'Numeric features      : {len(numeric_cols)}')
print(f'Categorical features  : {len(cat_cols)}')
print(f'Cột có missing values : {df_full.isnull().any().sum()}')
print(f'  → workclass         : {df_full["workclass"].isnull().sum():,} ({100*df_full["workclass"].isnull().mean():.2f}%)')
print(f'  → occupation        : {df_full["occupation"].isnull().sum():,} ({100*df_full["occupation"].isnull().mean():.2f}%)')
print(f'  → native_country    : {df_full["native_country"].isnull().sum():,} ({100*df_full["native_country"].isnull().mean():.2f}%)')
print(f'Duplicate rows        : {df_full.duplicated().sum():,}')
print(f'Class imbalance       : <=50K={100*(df_full["income"]=="<=50K").mean():.1f}%  >50K={100*(df_full["income"]==">50K").mean():.1f}%')
print()
print('📌 Các vấn đề cần xử lý ở bước Preprocessing:')
print('   1. Điền missing values (workclass, occupation, native_country)')
print('   2. Encode categorical features (8 cột object)')
print('   3. Scale numeric features (phân phối lệch ở capital_gain/loss)')
print('\n✅ EDA hoàn thành! Chạy 02_Preprocessing tiếp theo.')
